# DSC630 Predictive Analytics
## Week 10 Assignment - MovieLens Recommender System
### David Wederstrandt
#### 2026-03-01

## Recommender System Strategy

#### This notebook builds a content-based recommender that takes one movie title as input and returns 10 similar movies from the MovieLens dataset.

Why this strategy fits the assignment:
- The dataset provides rich item content (`genres` and user `tags`) for almost every movie.
- Content-based methods can recommend similar titles even without a target user's rating history.
- The approach is interpretable: recommendations can be explained by shared descriptive terms.

Pipeline overview:
1. Load and validate data files.
2. Build movie-level text features from genres and tags.
3. Convert text into TF-IDF vectors.
4. Compute pairwise cosine similarity between movies.
5. Re-rank candidates using rating quality and vote counts.
6. Return the top 10 recommendations for a user-selected movie.

The model is content-driven, with a lightweight rating-based rerank step to improve recommendation quality.

---

## Load Libraries
#### Import core packages for data processing, text vectorization, and similarity scoring.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from pathlib import Path
from difflib import get_close_matches
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

## Data Loading

### Read all MovieLens source files and verify dataset dimensions

In [2]:
# Load data

movies = pd.read_csv('data/movies.csv')
ratings = pd.read_csv('data/ratings.csv')
tags = pd.read_csv('data/tags.csv')
links = pd.read_csv('data/links.csv')

print(f'movies:  {movies.shape}')
print(f'ratings: {ratings.shape}')
print(f'tags:    {tags.shape}')
print(f'links:   {links.shape}')

movies:  (9742, 3)
ratings: (100836, 4)
tags:    (3683, 4)
links:   (9742, 3)


## Link Integrity Checks

#### Confirm that shared keys (`movieId`, `userId`) are consistent across files so joins and recommendations are reliable.

In [3]:
# Basic integrity checks for link keys
assert set(ratings['movieId']).issubset(set(movies['movieId']))
assert set(tags['movieId']).issubset(set(movies['movieId']))
assert set(links['movieId']) == set(movies['movieId'])
print('Key checks passed.')

Key checks passed.


## Feature Engineering

#### Build movie feature text by combining genres with aggregated user tags. This gives each movie a descriptive profile for similarity modeling.

In [4]:
# Aggregate tags per movie and build feature text
tag_text = (
    tags.groupby('movieId')['tag']
        .apply(lambda x: ' '.join(pd.Series(x).dropna().astype(str).str.lower()))
        .reset_index(name='tag_text')
)

model_df = movies.merge(tag_text, on='movieId', how='left')
model_df['tag_text'] = model_df['tag_text'].fillna('')
model_df['genres_text'] = model_df['genres'].str.replace('|', ' ', regex=False).str.lower()
model_df['feature_text'] = (model_df['genres_text'] + ' ' + model_df['tag_text']).str.strip()

# If a movie has no genres/tags, keep a placeholder token
model_df.loc[model_df['feature_text'].eq(''), 'feature_text'] = 'unknown'

model_df[['movieId', 'title', 'genres', 'feature_text']].head()

,movieId,title,genres,feature_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,adventure animation children comedy fantasy pi...
1,2,Jumanji (1995),Adventure|Children|Fantasy,adventure children fantasy fantasy magic board...
2,3,Grumpier Old Men (1995),Comedy|Romance,comedy romance moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,comedy drama romance
4,5,Father of the Bride Part II (1995),Comedy,comedy pregnancy remake


## Similarity Model Construction

#### Build TF-IDF vectors from feature text and compute cosine similarity across all movies.

**What is TF-IDF and why use it?**
- `TF` (term frequency) measures how strongly a word appears in one movie's feature text.
- `IDF` (inverse document frequency) downweights terms that appear in many movies and are less informative (for example, very common words).
- TF-IDF therefore emphasizes discriminative descriptors and reduces noise from ubiquitous tokens.

A standard intuition is:
- `tfidf(term, movie) = tf(term, movie) * idf(term)`

For this task, that matters because genres and tags are sparse text features; we want distinctive terms like `stop-motion`, `time travel`, or `pixar` to influence similarity more than generic terms.

**What is cosine similarity and why use it?**
- Cosine similarity compares the angle between two vectors:
- `cos(theta) = (A · B) / (||A|| ||B||)`
- Values near 1 mean very similar term composition; values near 0 mean little overlap.

Cosine works well with TF-IDF because it normalizes for vector length, so movies with many tags are not automatically favored over movies with fewer tags. It focuses on content profile shape rather than raw token count.

In [5]:
# Build TF-IDF matrix and cosine similarity
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(model_df['feature_text'])
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# Rating summary for reranking
rating_stats = ratings.groupby('movieId')['rating'].agg(['mean', 'count']).reset_index()
rating_stats.columns = ['movieId', 'rating_mean', 'rating_count']

print('TF-IDF matrix shape:', tfidf_matrix.shape)
print('Cosine similarity shape:', cosine_sim.shape)

TF-IDF matrix shape: (9742, 1677)
Cosine similarity shape: (9742, 9742)


## Recommendation Function

#### Define title resolution and recommendation logic. The final ranking combines text similarity with an IMDb-style weighted rating.

Note: Optimizer or ranking settings are documented directly in code comments for reproducibility.

In [6]:
# Helper indices for exact and fuzzy title lookup
title_to_index = pd.Series(model_df.index, index=model_df['title'].str.lower()).drop_duplicates()
all_titles_lower = model_df['title'].str.lower().tolist()

def resolve_title(query: str) -> str:
    """Return the best matching movie title from the dataset."""
    q = query.strip().lower()
    if q in title_to_index.index:
        return model_df.loc[title_to_index[q], 'title']

    # Fuzzy match if exact title is not found
    matches = get_close_matches(q, all_titles_lower, n=1, cutoff=0.6)
    if not matches:
        raise ValueError(f"Movie '{query}' not found in dataset. Try a different title.")

    best_match_lower = matches[0]
    best_idx = all_titles_lower.index(best_match_lower)
    return model_df.loc[best_idx, 'title']

In [7]:
def recommend_movies(movie_title: str, n_recommendations: int = 10, min_votes: int = 20, alpha: float = 0.75) -> pd.DataFrame:
    """
    Recommend movies similar to `movie_title`.

    Parameters
    ----------
    movie_title : str
        User-provided movie title.
    n_recommendations : int
        Number of movies to return.
    min_votes : int
        Minimum rating count preferred during reranking.
    alpha : float
        Weight on similarity score in final blend (0 to 1).
    """
    canonical_title = resolve_title(movie_title)
    idx = int(title_to_index[canonical_title.lower()])

    # Similarity scores for all movies vs selected movie
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:]  # remove the movie itself

    candidates = pd.DataFrame(sim_scores, columns=['index', 'similarity'])
    candidates = candidates.merge(
        model_df[['movieId', 'title', 'genres']].reset_index(),
        on='index',
        how='left'
    )
    candidates = candidates.merge(rating_stats, on='movieId', how='left')
    candidates['rating_mean'] = candidates['rating_mean'].fillna(0.0)
    candidates['rating_count'] = candidates['rating_count'].fillna(0).astype(int)

    # IMDb-style weighted rating to reduce bias toward low-vote movies
    C = ratings['rating'].mean()
    m = min_votes
    v = candidates['rating_count']
    R = candidates['rating_mean']
    candidates['weighted_rating'] = (v / (v + m)) * R + (m / (v + m)) * C

    # Normalize weighted rating to 0-1 and blend with similarity
    wr_min = candidates['weighted_rating'].min()
    wr_max = candidates['weighted_rating'].max()
    if wr_max > wr_min:
        candidates['weighted_rating_norm'] = (candidates['weighted_rating'] - wr_min) / (wr_max - wr_min)
    else:
        candidates['weighted_rating_norm'] = 0.0

    candidates['final_score'] = alpha * candidates['similarity'] + (1 - alpha) * candidates['weighted_rating_norm']

    # Prefer movies with at least min_votes; fallback if not enough
    preferred = candidates[candidates['rating_count'] >= min_votes].copy()
    pool = preferred if len(preferred) >= n_recommendations else candidates

    recs = (
        pool.sort_values(['final_score', 'similarity', 'rating_count'], ascending=False)
            .head(n_recommendations)
            .loc[:, ['title', 'genres', 'rating_mean', 'rating_count', 'similarity', 'final_score']]
            .reset_index(drop=True)
    )

    recs.insert(0, 'input_movie', canonical_title)
    return recs

## Example Recommendations

#### Generate top-10 recommendations for a known movie title and inspect result quality.

In [8]:
# Example (replace with any movie title from the dataset)
recommend_movies('Toy Story (1995)', n_recommendations=10)

,input_movie,title,genres,rating_mean,rating_count,similarity,final_score
0,Toy Story (1995),"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,3.516304,92,0.862225,0.779502
1,Toy Story (1995),Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.860825,97,0.644038,0.654777
2,Toy Story (1995),"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,3.871212,132,0.357912,0.443337
3,Toy Story (1995),Guardians of the Galaxy 2 (2017),Action|Adventure|Sci-Fi,3.925926,27,0.367650,0.440123
4,Toy Story (1995),Up (2009),Adventure|Animation|Children|Drama,4.004762,105,0.320916,0.429438
5,Toy Story (1995),Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,3.813953,43,0.347446,0.420807
6,Toy Story (1995),"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,3.716216,37,0.357912,0.418595
7,Toy Story (1995),Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,4.109091,55,0.293014,0.411621
8,Toy Story (1995),Ice Age (2002),Adventure|Animation|Children|Comedy,3.688235,85,0.313323,0.386758
9,Toy Story (1995),How to Train Your Dragon (2010),Adventure|Animation|Children|Fantasy|IMAX,3.943396,53,0.278807,0.383978


## Recommendation Diagnostics

#### Evaluate recommendation quality with simple quantitative checks: average similarity, vote support, and score range for the top-10 results.

In [9]:
# Diagnostics on the latest recommendation output
example_recs = recommend_movies('Toy Story (1995)', n_recommendations=10)

diag = pd.Series({
    'avg_similarity': example_recs['similarity'].mean(),
    'min_similarity': example_recs['similarity'].min(),
    'max_similarity': example_recs['similarity'].max(),
    'avg_rating_mean': example_recs['rating_mean'].mean(),
    'median_rating_count': example_recs['rating_count'].median(),
    'score_range': example_recs['final_score'].max() - example_recs['final_score'].min()
})

print('Recommendation diagnostics (Toy Story sample):')
print(diag.round(4))

Recommendation diagnostics (Toy Story sample):
avg_similarity          0.4143
min_similarity          0.2788
max_similarity          0.8622
avg_rating_mean         3.8450
median_rating_count    70.0000
score_range             0.3955
dtype: float64


In [18]:
# Interactive input
movie_you_like = input('\nEnter a movie title you like (exact or close match): ').strip()

if not movie_you_like:
    print('Please enter a movie title.')
else:
    try:
        recs = recommend_movies(movie_you_like, n_recommendations=10)
        print(f"\nRecommendations for: {recs.loc[0, 'input_movie']}")
        display(recs.drop(columns=['input_movie']))
    except ValueError as e:
        print(e)


Enter a movie title you like (exact or close match):  Animal House



Recommendations for: Animal House (1978)


,title,genres,rating_mean,rating_count,similarity,final_score
0,Monty Python's And Now for Something Completel...,Comedy,4.053571,28,0.149094,0.286859
1,Dazed and Confused (1993),Comedy,3.928571,42,0.149094,0.282400
2,"Planes, Trains & Automobiles (1987)",Comedy,3.851852,27,0.149094,0.270410
3,Tommy Boy (1995),Comedy,3.780000,50,0.149094,0.270091
4,Friday (1995),Comedy,3.775000,20,0.149094,0.261624
5,Bridesmaids (2011),Comedy,3.761905,21,0.149094,0.261165
6,Thank You for Smoking (2006),Comedy|Drama,4.027778,36,0.109494,0.259376
7,Election (1999),Comedy,3.660714,56,0.149094,0.258976
8,Four Rooms (1995),Comedy,3.700000,20,0.149094,0.256517
9,Waking Ned Devine (a.k.a. Waking Ned) (1998),Comedy,3.673077,26,0.149094,0.256207


## Model Performance Summary

The recommender successfully returns 10 movies for a user-selected title using content similarity from genres/tags, then improves ranking quality with a rating-based reliability adjustment.

Theoretical takeaway: TF-IDF + cosine similarity is a strong baseline for sparse text-based item profiles because it highlights discriminative terms and compares movies by normalized term composition rather than raw text length.

Key takeaways:
- Higher **similarity** indicates closer genre/tag alignment to the input movie.
- **rating_count** helps avoid unstable rankings driven by very low-vote titles.
- **final_score** balances content similarity and rating reliability.

Limitation: this approach does not use personalized user history, so recommendations are similar-content suggestions rather than user-specific collaborative filtering results.

## Sources

1. GroupLens MovieLens dataset documentation (schema and dataset description):
   - https://grouplens.org/datasets/movielens/
2. Scikit-learn TF-IDF vectorization:
   - https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
3. Scikit-learn cosine similarity and linear kernel usage:
   - https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity
4. Content-based recommendation overview:
   - Ricci, F., Rokach, L., & Shapira, B. (Eds.). (2015). Recommender systems handbook (2nd ed.). Springer.